# Metode CBOW Pada Berita CNN

# Upgrade Library Gensim

Pertama-tama, kita memastikan library Gensim yang digunakan untuk membuat model Word2Vec adalah versi terbaru.

In [ ]:
!pip install --upgrade gensim

# Import Library

Setelah meng-upgrade, kita mengimpor library yang dibutuhkan

In [ ]:
import pandas as pd
from gensim.models import Word2Vec
import numpy as np


# Load File Dataset

Selanjutnya, dataset berita yang sudah di-stemmed dimuat dari file CSV. Kita menampilkan beberapa baris pertama untuk memastikan data terbaca dengan benar.

In [ ]:
# Load CSV dari Google Drive
df = pd.read_csv('/content/drive/MyDrive/PPW/berita/cnn_berita_isi_stemmed(baru).csv')

# Cek kolom (misalnya namanya 'abstrak' atau 'text')
df.head()

,Isi,cleaned,Kategori,no_stopwords,stemmed
0,"Jakarta, CNN Indonesia --\n ...",jakarta cnn indonesia sejumlah pengemudi ojek ...,Nasional,jakarta cnn indonesia sejumlah pengemudi ojek ...,jakarta cnn indonesia jumlah kemudi ojek onlin...
1,"Jakarta, CNN Indonesia -- Jenderal TNI (purn) ...",jakarta cnn indonesia jenderal tni purn djamar...,Nasional,jakarta cnn indonesia jenderal tni purn djamar...,jakarta cnn indonesia jenderal tni purn djamar...
2,"Jakarta, CNN Indonesia -- Korban meninggal dun...",jakarta cnn indonesia korban meninggal dunia k...,Nasional,jakarta cnn indonesia korban meninggal dunia k...,jakarta cnn indonesia korban tinggal dunia cel...
3,"Jakarta, CNN Indonesia -- Rapat evaluasi sekal...",jakarta cnn indonesia rapat evaluasi sekaligus...,Nasional,jakarta cnn indonesia rapat evaluasi sekaligus...,jakarta cnn indonesia rapat evaluasi sekaligus...
4,"Jakarta, CNN Indonesia -- Presiden Prabowo Sub...",jakarta cnn indonesia presiden prabowo subiant...,Nasional,jakarta cnn indonesia presiden prabowo subiant...,jakarta cnn indonesia presiden prabowo subiant...


In [ ]:
df[['Isi', 'stemmed','Kategori']].head(10)

,Isi,stemmed,Kategori
0,"Jakarta, CNN Indonesia --\n ...",jakarta cnn indonesia jumlah kemudi ojek onlin...,Nasional
1,"Jakarta, CNN Indonesia -- Jenderal TNI (purn) ...",jakarta cnn indonesia jenderal tni purn djamar...,Nasional
2,"Jakarta, CNN Indonesia -- Korban meninggal dun...",jakarta cnn indonesia korban tinggal dunia cel...,Nasional
3,"Jakarta, CNN Indonesia -- Rapat evaluasi sekal...",jakarta cnn indonesia rapat evaluasi sekaligus...,Nasional
4,"Jakarta, CNN Indonesia -- Presiden Prabowo Sub...",jakarta cnn indonesia presiden prabowo subiant...,Nasional
5,"Jakarta, CNN Indonesia -- Boyamin Saiman, peng...",jakarta cnn indonesia boyamin saiman acara kel...,Nasional
6,"Jakarta, CNN Indonesia -- Komisi Pemberantasan...",jakarta cnn indonesia komisi berantas korupsi ...,Nasional
7,"Jakarta, CNN Indonesia -- Pengemudi ojek onlin...",jakarta cnn indonesia kemudi ojek online ojol ...,Nasional
8,"Jakarta, CNN Indonesia -- Wakil Ketua Komisi X...",jakarta cnn indonesia wakil ketua komisi xiii ...,Nasional
9,"Jakarta, CNN Indonesia -- Pengemudi ojek onlin...",jakarta cnn indonesia kemudi ojek online ojol ...,Nasional


# Tokenisasi teks dan representasi vektor rata-rata menggunakan Word2Vec

Setelah data siap, kita membuat tokenizer sederhana untuk memisahkan kata-kata dan vectorizer khusus yang akan mengubah tiap dokumen menjadi vektor rata-rata kata berdasarkan model Word2Vec.

1. Tokenizer: mengubah teks menjadi lowercase dan memisahkan kata-kata.

2. MeanEmbeddingVectorizer: menghitung rata-rata vektor kata tiap dokumen; jika dokumen tidak memiliki kata dalam vocabulary Word2Vec, maka digunakan vektor nol.

In [ ]:
import numpy as np

class MyTokenizer:
    def fit_transform(self, texts):
        # Tokenisasi sederhana: lowercase + split
        return [str(text).lower().split() for text in texts]

class MeanEmbeddingVectorizer:
    def __init__(self, word2vec_model):
        self.word2vec = word2vec_model
        # Perbaikan: gunakan vector_size (Gensim ≥ 4.0)
        self.dim = word2vec_model.wv.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_tokenized = MyTokenizer().fit_transform(X)
        embeddings = []
        for words in X_tokenized:
            # Ambil vektor hanya untuk kata yang ada di vocab
            valid_vectors = [
                self.word2vec.wv[word] for word in words
                if word in self.word2vec.wv
            ]
            if valid_vectors:
                embeddings.append(np.mean(valid_vectors, axis=0))
            else:
                embeddings.append(np.zeros(self.dim))
        return np.array(embeddings)

    def fit_transform(self, X, y=None):
        return self.transform(X)

In [ ]:
df.shape

(187, 5)

# Word2Vec Embedding dari Korpus

Dataset yang sudah di-stemmed diubah menjadi corpus, yaitu daftar kata-kata per dokumen. Setiap baris teks dipecah menjadi kata-kata menggunakan spasi sebagai pemisah.

In [ ]:
corpus = []
for col in df.stemmed:
   word_list = col.split(" ")
   corpus.append(word_list)

#show first value
corpus[0:1]

#generate vectors from corpus
model = Word2Vec(corpus, min_count=1, vector_size = 56)

hasil corpus

In [ ]:
print(list(model.wv.key_to_index.keys())[:50])  # 50 kata pertama di vocab


['indonesia', 'sebut', 'jadi', 'cnn', 'lihat', 'jakarta', 'kata', 'menteri', 'laku', 'lebih', 'tahun', 'with', 'negara', 'content', 'to', 'scroll', 'continue', 'advertisement', 'video', 'israel', 'hingga', 'gambas', 'lalu', 'orang', 'satu', 'rp', 'baru', 'buat', 'itu', 'the', 'ada', 'masuk', 'guna', 'besar', 'presiden', 'jumlah', 'belum', 'rabu', 'sama', 'hari', 'jabat', 'unit', 'tak', 'jalan', 'anak', 'banyak', 'kerja', 'perintah', 'lama', 'ujar']


# Eksplorasi Word2Vec dan Mean

Setelah model Word2Vec dibuat, kita bisa mengeksplorasi hubungan antar kata:
1. Menampilkan kata-kata yang paling mirip dengan kata 'indonesia' berdasarkan cosine similarity.
2. Menampilkan kata yang paling cocok jika kita menambahkan 'presiden' dan 'negara' tapi mengurangi pengaruh 'menteri'.

In [ ]:
#explore embeddings using cosine similarity
model.wv.most_similar('indonesia')

model.wv.most_similar_cosmul(positive = ['presiden', 'negara'], negative = ['menteri'])

model.wv.doesnt_match("indonesia jakarta presiden rp".split())

#save embeddings
filename = '/content/drive/MyDrive/PPW/berita/berita_embd.txt'
model.wv.save_word2vec_format(filename, binary=False)


setelah itu mengubah dokumen menjadi vektor rata rata

In [ ]:
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
mean_embedded = mean_embedding_vectorizer.fit_transform(df['stemmed'])

# Hasil Embedding Dokumen

Setelah membuat mean embeddings untuk tiap dokumen, langkah selanjutnya adalah menyimpan hasilnya ke dalam kolom baru di DataFrame. Kolom baru ini, yang dinamai 'array', berisi vektor numerik yang merepresentasikan setiap dokumen sebagai rata-rata vektor kata-kata di dalamnya.

In [ ]:
df['array']=list(mean_embedded)
# hanya ambil kolom yang dibutuhkan
df_selected = df[['Isi', 'stemmed','array', 'Kategori']]

# tampilkan 5 baris pertama
df_selected.head(5)

,Isi,stemmed,array,Kategori
0,"Jakarta, CNN Indonesia --\n ...",jakarta cnn indonesia jumlah kemudi ojek onlin...,"[-0.1446159, 0.32593662, 0.23336366, 0.0930218...",Nasional
1,"Jakarta, CNN Indonesia -- Jenderal TNI (purn) ...",jakarta cnn indonesia jenderal tni purn djamar...,"[-0.10747767, 0.2925398, 0.19437744, 0.0693292...",Nasional
2,"Jakarta, CNN Indonesia -- Korban meninggal dun...",jakarta cnn indonesia korban tinggal dunia cel...,"[-0.09961995, 0.24957089, 0.17018116, 0.069914...",Nasional
3,"Jakarta, CNN Indonesia -- Rapat evaluasi sekal...",jakarta cnn indonesia rapat evaluasi sekaligus...,"[-0.104498565, 0.2775947, 0.18800326, 0.070568...",Nasional
4,"Jakarta, CNN Indonesia -- Presiden Prabowo Sub...",jakarta cnn indonesia presiden prabowo subiant...,"[-0.11487023, 0.2885689, 0.20686294, 0.0790582...",Nasional


# Panjang Vektor Embedding per Dokumen

Setelah menyimpan vektor dokumen di kolom 'array', langkah berikutnya adalah memeriksa dimensi atau panjang vektor setiap dokumen

In [ ]:
df['embedding_length'] = df['array'].str.len()

In [ ]:
print(df['embedding_length'])

0      56
1      56
2      56
3      56
4      56
       ..
182    56
183    56
184    56
185    56
186    56
Name: embedding_length, Length: 187, dtype: int64


Hasil df.shape menunjukkan (187, 7), yang berarti DataFrame memiliki 187 baris dan 7 kolom. Ini mengindikasikan bahwa dataset terdiri dari 187 dokumen, dan setiap dokumen sekarang memiliki 7 atribut

In [ ]:
df.shape

(187, 7)

# list embedding

Langkah ini bertujuan untuk mengubah kolom vektor embedding yang berupa list menjadi format DataFrame dengan satu kolom per dimensi vektor.

In [ ]:
num_features = len(df['array'].iloc[0])  # asumsi semua list punya panjang sama
columns = [f'f{i+1}' for i in range(num_features)]

# Inisialisasi dictionary untuk menampung data per kolom
data_dict = {col: [] for col in columns}

# Looping setiap baris di kolom 'embedding'
for embedding_list in df['array']:
    for i, value in enumerate(embedding_list):
        data_dict[f'f{i+1}'].append(value)

# Buat DataFrame dari dictionary
embedding_df = pd.DataFrame(data_dict)

embedding_df

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f47,f48,f49,f50,f51,f52,f53,f54,f55,f56
0,-0.144616,0.325937,0.233364,0.093022,0.219246,-0.383782,0.312154,-0.816783,-0.237540,-0.408026,...,0.077216,0.325907,0.000709,0.166409,-0.102221,0.589225,0.337356,0.313535,-0.110673,-0.245294
1,-0.107478,0.292540,0.194377,0.069329,0.173760,-0.323973,0.264133,-0.641519,-0.219561,-0.326198,...,0.070582,0.244173,0.007690,0.136839,-0.076619,0.469327,0.279003,0.258706,-0.102409,-0.176039
2,-0.099620,0.249571,0.170181,0.069914,0.154348,-0.280293,0.237866,-0.584662,-0.186185,-0.292252,...,0.057720,0.226143,0.003912,0.112535,-0.068443,0.424419,0.257370,0.230665,-0.092301,-0.167768
3,-0.104499,0.277595,0.188003,0.070569,0.164971,-0.307924,0.255527,-0.618967,-0.208577,-0.311481,...,0.068948,0.231163,0.008110,0.125979,-0.078600,0.452833,0.271901,0.250729,-0.095983,-0.170570
4,-0.114870,0.288569,0.206863,0.079058,0.184993,-0.328868,0.275398,-0.676313,-0.218100,-0.340606,...,0.062208,0.253764,0.006716,0.130214,-0.081101,0.492066,0.291201,0.264792,-0.099421,-0.194764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,-0.118394,0.282082,0.192755,0.075929,0.176885,-0.321007,0.265792,-0.659153,-0.213053,-0.329235,...,0.068746,0.251830,0.005606,0.132081,-0.076068,0.478415,0.281986,0.261348,-0.100545,-0.188710
183,-0.078652,0.202167,0.136312,0.052057,0.119766,-0.225049,0.183403,-0.451074,-0.153007,-0.226170,...,0.048872,0.170281,0.005804,0.094147,-0.055870,0.329721,0.193799,0.180765,-0.071726,-0.121635
184,-0.094375,0.257816,0.172751,0.063335,0.146028,-0.284931,0.226862,-0.556503,-0.195082,-0.282706,...,0.067181,0.206149,0.010460,0.123643,-0.072070,0.408289,0.237357,0.227473,-0.088372,-0.148892
185,-0.090590,0.230753,0.157081,0.063016,0.137156,-0.258390,0.212644,-0.522014,-0.175160,-0.262010,...,0.056780,0.198427,0.005570,0.108839,-0.066231,0.381647,0.226769,0.208726,-0.082894,-0.142669


Setelah membuat DataFrame embedding_df yang berisi vektor embedding tiap dokumen dengan satu kolom per dimensi, kita menambahkan kolom 'Kategori' dari DataFrame asli.

In [ ]:
embedding_df['Kategori'] = df['Kategori'].values

In [ ]:
embedding_df

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f48,f49,f50,f51,f52,f53,f54,f55,f56,Kategori
0,-0.144616,0.325937,0.233364,0.093022,0.219246,-0.383782,0.312154,-0.816783,-0.237540,-0.408026,...,0.325907,0.000709,0.166409,-0.102221,0.589225,0.337356,0.313535,-0.110673,-0.245294,Nasional
1,-0.107478,0.292540,0.194377,0.069329,0.173760,-0.323973,0.264133,-0.641519,-0.219561,-0.326198,...,0.244173,0.007690,0.136839,-0.076619,0.469327,0.279003,0.258706,-0.102409,-0.176039,Nasional
2,-0.099620,0.249571,0.170181,0.069914,0.154348,-0.280293,0.237866,-0.584662,-0.186185,-0.292252,...,0.226143,0.003912,0.112535,-0.068443,0.424419,0.257370,0.230665,-0.092301,-0.167768,Nasional
3,-0.104499,0.277595,0.188003,0.070569,0.164971,-0.307924,0.255527,-0.618967,-0.208577,-0.311481,...,0.231163,0.008110,0.125979,-0.078600,0.452833,0.271901,0.250729,-0.095983,-0.170570,Nasional
4,-0.114870,0.288569,0.206863,0.079058,0.184993,-0.328868,0.275398,-0.676313,-0.218100,-0.340606,...,0.253764,0.006716,0.130214,-0.081101,0.492066,0.291201,0.264792,-0.099421,-0.194764,Nasional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,-0.118394,0.282082,0.192755,0.075929,0.176885,-0.321007,0.265792,-0.659153,-0.213053,-0.329235,...,0.251830,0.005606,0.132081,-0.076068,0.478415,0.281986,0.261348,-0.100545,-0.188710,Gaya Hidup
183,-0.078652,0.202167,0.136312,0.052057,0.119766,-0.225049,0.183403,-0.451074,-0.153007,-0.226170,...,0.170281,0.005804,0.094147,-0.055870,0.329721,0.193799,0.180765,-0.071726,-0.121635,Gaya Hidup
184,-0.094375,0.257816,0.172751,0.063335,0.146028,-0.284931,0.226862,-0.556503,-0.195082,-0.282706,...,0.206149,0.010460,0.123643,-0.072070,0.408289,0.237357,0.227473,-0.088372,-0.148892,Gaya Hidup
185,-0.090590,0.230753,0.157081,0.063016,0.137156,-0.258390,0.212644,-0.522014,-0.175160,-0.262010,...,0.198427,0.005570,0.108839,-0.066231,0.381647,0.226769,0.208726,-0.082894,-0.142669,Gaya Hidup


Hasil embedding_df.shape menunjukkan (187, 57), yang berarti DataFrame memiliki 187 baris dan 57 kolom. Artinya, terdapat 187 dokumen yang masing-masing direpresentasikan sebagai vektor numerik berdimensi 56 (kolom f1 sampai f56) ditambah satu kolom tambahan untuk kategori dokumen.

In [ ]:
embedding_df.shape

(187, 57)

# Label Encorder Kategori

Untuk mempersiapkan data kategori agar bisa digunakan dalam algoritma machine learning, kita mengubah label kategori berbentuk teks menjadi angka menggunakan LabelEncoder dari scikit-learn. Setiap kategori teks dipetakan ke nilai numerik unik, dan hasilnya disimpan di kolom baru 'Kategori_encoded'.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# encode kategori teks -> angka
le = LabelEncoder()
embedding_df['Kategori_encoded'] = le.fit_transform(embedding_df['Kategori'])

# Cek mapping kategori -> angka
print(dict(zip(le.classes_, le.transform(le.classes_))))


{'Ekonomi': 0, 'Gaya Hidup': 1, 'Hiburan': 2, 'Internasional': 3, 'Nasional': 4, 'Otomotif': 5, 'Teknologi': 6}


In [ ]:
embedding_df_encoded = embedding_df.drop(columns=['Kategori'])
embedding_df_encoded.head(10)


,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f48,f49,f50,f51,f52,f53,f54,f55,f56,Kategori_encoded
0,-0.144616,0.325937,0.233364,0.093022,0.219246,-0.383782,0.312154,-0.816783,-0.237540,-0.408026,...,0.325907,0.000709,0.166409,-0.102221,0.589225,0.337356,0.313535,-0.110673,-0.245294,4
1,-0.107478,0.292540,0.194377,0.069329,0.173760,-0.323973,0.264133,-0.641519,-0.219561,-0.326198,...,0.244173,0.007690,0.136839,-0.076619,0.469327,0.279003,0.258706,-0.102409,-0.176039,4
2,-0.099620,0.249571,0.170181,0.069914,0.154348,-0.280293,0.237866,-0.584662,-0.186185,-0.292252,...,0.226143,0.003912,0.112535,-0.068443,0.424419,0.257370,0.230665,-0.092301,-0.167768,4
3,-0.104499,0.277595,0.188003,0.070569,0.164971,-0.307924,0.255527,-0.618967,-0.208577,-0.311481,...,0.231163,0.008110,0.125979,-0.078600,0.452833,0.271901,0.250729,-0.095983,-0.170570,4
4,-0.114870,0.288569,0.206863,0.079058,0.184993,-0.328868,0.275398,-0.676313,-0.218100,-0.340606,...,0.253764,0.006716,0.130214,-0.081101,0.492066,0.291201,0.264792,-0.099421,-0.194764,4
5,-0.099589,0.250858,0.172306,0.068904,0.154951,-0.280872,0.235712,-0.579883,-0.189697,-0.292782,...,0.222714,0.003507,0.113423,-0.069355,0.420900,0.251609,0.231627,-0.087442,-0.163745,4
6,-0.100659,0.245129,0.168322,0.068292,0.154955,-0.279446,0.234112,-0.580005,-0.184611,-0.291386,...,0.224404,0.003197,0.112532,-0.067128,0.422224,0.254319,0.228136,-0.086615,-0.166388,4
7,-0.124455,0.316724,0.217504,0.086412,0.194507,-0.359935,0.297774,-0.728015,-0.239007,-0.365275,...,0.276956,0.004872,0.145168,-0.089025,0.533011,0.317936,0.291978,-0.112317,-0.207543,4
8,-0.100740,0.248753,0.170126,0.068232,0.154862,-0.279815,0.237056,-0.582843,-0.186932,-0.291687,...,0.221486,0.002279,0.112321,-0.070010,0.423926,0.257315,0.230869,-0.089666,-0.166074,4
9,-0.124455,0.316724,0.217504,0.086412,0.194507,-0.359935,0.297774,-0.728015,-0.239007,-0.365275,...,0.276956,0.004872,0.145168,-0.089025,0.533011,0.317936,0.291978,-0.112317,-0.207543,4


# Modelling

Setelah data embedding siap dan kategori diubah menjadi angka, kita dapat melakukan modeling klasifikasi untuk memprediksi kategori dokumen.

##Random Forest

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ===========================
# Asumsi embedding_df sudah ada
# Kolom: f1, f2, ..., fN (numerik) + 'Kategori' (opsional) + 'Kategori_encoded'
# ===========================

# Pastikan kolom fitur hanya numerik
X = embedding_df.drop(columns=['Kategori_encoded', 'Kategori'], errors='ignore')
y = embedding_df['Kategori_encoded']

# Split data train & test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Inisialisasi model Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Prediksi
y_pred = model.predict(X_test)

# Evaluasi
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.5526315789473685
              precision    recall  f1-score   support

           0       0.43      0.50      0.46         6
           1       0.75      0.60      0.67         5
           2       0.60      0.50      0.55         6
           3       0.67      0.67      0.67         6
           4       0.57      0.67      0.62         6
           5       0.40      0.50      0.44         4
           6       0.50      0.40      0.44         5

    accuracy                           0.55        38
   macro avg       0.56      0.55      0.55        38
weighted avg       0.56      0.55      0.55        38



Hasil Random Forest menunjukkan akurasi sekitar 55%, artinya model benar memprediksi sedikit lebih dari separuh dokumen di test set. Beberapa kelas, seperti kelas 1 dan 3, memiliki precision dan recall lebih tinggi, sedangkan kelas lain performanya rendah. Secara keseluruhan, model masih cukup menengah dan bisa ditingkatkan dengan menambah data, mengoptimalkan hyperparameter, atau mencoba embedding/dimensi lain.

##Naive Bayes

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# ===========================
# Asumsi embedding_df sudah ada
# Kolom: f1, f2, ..., fN (numerik) + 'Kategori_encoded'
# ===========================

# Pisahkan fitur & target
X = embedding_df.drop(columns=['Kategori_encoded', 'Kategori'], errors='ignore')
y = embedding_df['Kategori_encoded']

# Split data train & test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Inisialisasi model Gaussian Naive Bayes
model = GaussianNB()
model.fit(X_train, y_train)

# Prediksi
y_pred = model.predict(X_test)

# Evaluasi
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.2894736842105263
              precision    recall  f1-score   support

           0       0.33      0.50      0.40         6
           1       0.25      0.20      0.22         5
           2       0.20      0.33      0.25         6
           3       0.40      0.33      0.36         6
           4       0.00      0.00      0.00         6
           5       0.60      0.75      0.67         4
           6       0.00      0.00      0.00         5

    accuracy                           0.29        38
   macro avg       0.25      0.30      0.27        38
weighted avg       0.24      0.29      0.26        38



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Hasil Naive Bayes menunjukkan akurasi yang lebih rendah, sekitar 29%, artinya model hanya berhasil memprediksi kurang dari sepertiga dokumen di test set dengan benar. Beberapa kelas, seperti kelas 5, memiliki precision dan recall tinggi (precision 0,6, recall 0,75), tetapi banyak kelas lain, terutama kelas 4 dan 6, tidak terprediksi sama sekali (precision dan recall 0). Secara keseluruhan, model Naive Bayes kurang efektif untuk dataset ini dibanding Random Forest, kemungkinan karena jumlah data kecil dan distribusi kata pada embedding tidak cocok dengan asumsi independensi Naive Bayes.